In [1]:
import pandas as pd
from tqdm.auto import tqdm
import sys
sys.path.append("../")
from dotenv import load_dotenv
load_dotenv()
tqdm.pandas()

In [2]:
MODEL_NAME = "gpt-5.2-2025-12-11"
CONTEXT_WINDOW = 395000 # gpt 5.1's context window is 400k tokens

In [3]:
# Import double-coded annotations - used as ground truth to evaluate model performance

df_annotations = pd.read_csv("../data/7_annotations_repos/repo_annotations.csv")

In [4]:
# create a copy of df_annotations where every column except PMCID and repo_url
# has "_gt" appended to its name
cols_to_rename = [c for c in df_annotations.columns if c not in ("PMCID", "repo_url")]
rename_map = {c: f"{c}_gt" for c in cols_to_rename}

df_repos_assessment = df_annotations.copy().rename(columns=rename_map)

In [5]:
from src.repo_utils import clone_and_extract_tree
from src.llm_utils import LLM_wrapper, RepoAssessment
from src.llm_utils import REPO_ASSESSMENT_PROMPT_v1 as repo_assessment_prompt

In [6]:
df_annotations.head()

,PMCID,repo_url,is_empty,contains_readme,readme_purpose_and_outputs,contains_requirements,requirements_dependency_versions,contains_license,sufficient_code_documentation,is_modular_and_structured,implements_tests,fixes_seed_if_stochastic,hardware_requirements,contains_link_to_paper,contains_citation,includes_data_or_sample
0,PMC11307559,https://seakheeoh76.shinyapps.io/XGBoost_BA_pr...,True,False,False,False,False,False,False,False,False,False,False,False,False,False
1,PMC12121029,https://github.com/EnzhaoZhu/Common-risks-pred...,False,True,True,True,True,False,False,True,False,False,False,False,False,False
2,PMC11701130,https://github.com/CalvinSMU/AiLES,True,True,False,False,False,False,False,False,False,False,False,True,True,False
3,PMC8175333,https://github.com/ATARIQ5,True,False,False,False,False,False,False,False,False,False,False,False,False,False
4,PMC8932490,https://github.com/Aneja-Lab-Yale/Aneja-Lab-Pu...,False,True,True,True,True,False,True,True,False,False,False,True,False,False


In [7]:
def wrapper_clone_and_extract_tree(repo_url):
    try:
        return clone_and_extract_tree(repo_url=repo_url,
                                      context_window=CONTEXT_WINDOW,
                                      clone_dir="../data/cloned_repos")
    except Exception as e:
        return f"ERROR: {e}"

df_repos_assessment["repo_content"] = df_repos_assessment["repo_url"].progress_apply(wrapper_clone_and_extract_tree)

  0%|          | 0/35 [00:00<?, ?it/s]

Repo already exists at ../data/cloned_repos/Common-risks-prediction-among-psychiatric-inpatients. Skipping clone.
Repo already exists at ../data/cloned_repos/AiLES. Skipping clone.
Repo already exists at ../data/cloned_repos/Aneja-Lab-Public-Adversarial-Imaging. Skipping clone.
Repo already exists at ../data/cloned_repos/champCalculator. Skipping clone.
Repo already exists at ../data/cloned_repos/Prediction_of_CDI_by_EHR_and_Genetics. Skipping clone.
Repo already exists at ../data/cloned_repos/MLSPE. Skipping clone.
Repo already exists at ../data/cloned_repos/EmcDementiaModelValidation. Skipping clone.
Repo already exists at ../data/cloned_repos/Pathomics-analysis. Skipping clone.
Zenodo repo already exists and is not empty at ../data/cloned_repos/zenodo. Skipping download.
Repo already exists at ../data/cloned_repos/Emergency-data-processing. Skipping clone.
Repo already exists at ../data/cloned_repos/Predict-Prostate. Skipping clone.
OSF repo already exists and is not empty at ../dat

In [8]:
df_repos_assessment.to_csv("../data/8_repo_assessment/repo_assessment_gt_and_repo_content.csv", index=False)

In [9]:
llm_wrapper = LLM_wrapper(
    model_name=MODEL_NAME,
    system_prompt=repo_assessment_prompt,
    output_format_class=RepoAssessment
)

In [10]:
df_pred = llm_wrapper.assess_dataframe(
    input_file_path="../data/8_repo_assessment/repo_assessment_gt_and_repo_content.csv",
    text_column="repo_content",
    output_dir="../data/8_repo_assessment/"
)

Total rows: 35, Already processed: 0, Remaining: 35


100%|██████████| 35/35 [02:06<00:00,  3.62s/it]


In [11]:
df_pred.columns

Index(['PMCID', 'repo_url', 'is_empty_gt', 'contains_readme_gt',
       'readme_purpose_and_outputs_gt', 'contains_requirements_gt',
       'requirements_dependency_versions_gt', 'contains_license_gt',
       'sufficient_code_documentation_gt', 'is_modular_and_structured_gt',
       'implements_tests_gt', 'fixes_seed_if_stochastic_gt',
       'hardware_requirements_gt', 'contains_link_to_paper_gt',
       'contains_citation_gt', 'includes_data_or_sample_gt', 'repo_content',
       'generation', 'is_empty_pred', 'contains_readme_pred',
       'readme_purpose_and_outputs_pred', 'contains_requirements_pred',
       'requirements_dependency_versions_pred', 'contains_license_pred',
       'sufficient_code_documentation_pred', 'is_modular_and_structured_pred',
       'implements_tests_pred', 'fixes_seed_if_stochastic_pred',
       'hardware_requirements_pred', 'contains_link_to_paper_pred',
       'contains_citation_pred', 'includes_data_or_sample_pred'],
      dtype='object')

In [13]:
df_repos_assessment

,PMCID,repo_url,is_empty_gt,contains_readme_gt,readme_purpose_and_outputs_gt,contains_requirements_gt,requirements_dependency_versions_gt,contains_license_gt,sufficient_code_documentation_gt,is_modular_and_structured_gt,implements_tests_gt,fixes_seed_if_stochastic_gt,hardware_requirements_gt,contains_link_to_paper_gt,contains_citation_gt,includes_data_or_sample_gt,repo_content
0,PMC11307559,https://seakheeoh76.shinyapps.io/XGBoost_BA_pr...,True,False,False,False,False,False,False,False,False,False,False,False,False,False,None
1,PMC12121029,https://github.com/EnzhaoZhu/Common-risks-pred...,False,True,True,True,True,False,False,True,False,False,False,False,False,False,"Repository tree \n {\n ""url"": ""https://github..."
2,PMC11701130,https://github.com/CalvinSMU/AiLES,True,True,False,False,False,False,False,False,False,False,False,True,True,False,"Repository tree \n {\n ""url"": ""https://github..."
3,PMC8175333,https://github.com/ATARIQ5,True,False,False,False,False,False,False,False,False,False,False,False,False,False,None
4,PMC8932490,https://github.com/Aneja-Lab-Yale/Aneja-Lab-Pu...,False,True,True,True,True,False,True,True,False,False,False,True,False,False,"Repository tree \n {\n ""url"": ""https://github..."
5,PMC11040883,https://github.com/laamit/champCalculator,False,True,True,True,True,True,True,True,True,False,False,False,False,True,"Repository tree \n {\n ""url"": ""https://github..."
6,PMC10545794,https://github.com/TheDecodeLab/Prediction_of_...,False,True,False,False,False,True,False,False,False,True,False,False,False,False,"Repository tree \n {\n ""url"": ""https://github..."
7,PMC10658003,https://github.com/IreneSophia/MLSPE,False,True,True,False,False,False,True,True,False,False,False,False,True,False,"Repository tree \n {\n ""url"": ""https://github..."
8,PMC9720950,https://github.com/mi-erasmusmc/EmcDementiaMod...,False,True,True,False,False,False,False,True,False,False,False,False,False,False,"Repository tree \n {\n ""url"": ""https://github..."
9,PMC9653436,https://github.com/Dexin-Chen/Pathomics-analysis,False,False,False,False,False,False,False,False,False,True,False,False,False,False,"Repository tree \n {\n ""url"": ""https://github..."


In [15]:
# Repos not processed

df_repos_assessment[df_repos_assessment["repo_content"].isna()][["PMCID", "repo_url"]]

,PMCID,repo_url
0,PMC11307559,https://seakheeoh76.shinyapps.io/XGBoost_BA_pr...
3,PMC8175333,https://github.com/ATARIQ5
13,PMC9685866,https://doi.org/10.57770/QTYQZS
19,PMC11992218,https://www.cdc.gov/nchs/nhanes/index.htm
21,PMC10844815,https://github.com/yiwangz/SS_learning/MLinHCT
30,PMC8784888,https://github.com/AkihiroShiroshita/Predictio...


In [16]:
# Count rows where repo_content is not null
n_with_content = df_repos_assessment["repo_content"].notna().sum()
print(f"Number of rows with repo content: {n_with_content}")

Number of rows with repo content: 29


In [17]:
import numpy as np
import pandas as pd

fields = [
    "is_empty",
    "contains_readme",
    "readme_purpose_and_outputs",
    "contains_requirements",
    "requirements_dependency_versions",
    "contains_license",
    "sufficient_code_documentation",
    "is_modular_and_structured",
    "implements_tests",
    "fixes_seed_if_stochastic",
    "hardware_requirements",
    "contains_link_to_paper",
    "contains_citation",
    "includes_data_or_sample",
]

metrics = []

for field in fields:
    gt_col = f"{field}_gt"
    pred_col = f"{field}_pred"

    # Drop rows where either GT or prediction is NaN for this field
    sub = df_pred[[gt_col, pred_col]].dropna()

    if sub.empty:
        continue

    y_true = sub[gt_col].astype(bool)
    y_pred = sub[pred_col].astype(bool)

    tp = ((y_true) & (y_pred)).sum()
    tn = ((~y_true) & (~y_pred)).sum()
    fp = ((~y_true) & (y_pred)).sum()
    fn = ((y_true) & (~y_pred)).sum()

    accuracy = (y_true == y_pred).mean()
    recall_true = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    precision_true = tp / (tp + fp) if (tp + fp) > 0 else np.nan

    metrics.append(
        {
            "field": field,
            "n_non_null": len(sub),
            "accuracy": accuracy,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "recall_true": recall_true,
            "precision_true": precision_true,
        }
    )

metrics_df = pd.DataFrame(metrics).set_index("field")
metrics_df

,n_non_null,accuracy,tp,tn,fp,fn,recall_true,precision_true
field,,,,,,,,
is_empty,29,0.965517,1,27,1,0,1.000000,0.500000
contains_readme,29,0.965517,23,5,0,1,0.958333,1.000000
readme_purpose_and_outputs,23,0.739130,11,6,4,2,0.846154,0.733333
contains_requirements,29,0.793103,5,18,5,1,0.833333,0.500000
requirements_dependency_versions,6,0.833333,5,0,1,0,1.000000,0.833333
contains_license,29,0.965517,7,21,0,1,0.875000,1.000000
sufficient_code_documentation,29,0.689655,8,12,5,4,0.666667,0.615385
is_modular_and_structured,29,0.827586,15,9,1,4,0.789474,0.937500
implements_tests,29,1.000000,1,28,0,0,1.000000,1.000000


In [22]:
fp_is_empty = df_pred.loc[
    (df_pred["is_empty_pred"] == True) &
    (df_pred["is_empty_gt"] == False),
    ["PMCID", "repo_url", "is_empty_pred", "is_empty_gt", "repo_content"]
]

print(fp_is_empty.to_string(index=False))

     PMCID                               repo_url is_empty_pred  is_empty_gt                                                                                                                                                                                                                                                                        repo_content
PMC8881932 https://doi.org/10.5281/zenodo.4560078          True        False Repository tree \n {\n  "url": "https://doi.org/10.5281/zenodo.4560078",\n  "tree": [\n    {\n      "name": "epiben",\n      "children": [\n        {\n          "name": "hidosfaikid-v1.0.2.zip",\n          "children": null\n        }\n      ]\n    }\n  ]\n}\nFiles content\n 


In [24]:
column_name = "contains_readme"
fp_is_empty = df_pred.loc[
    (df_pred[f"{column_name}_pred"] == False) &
    (df_pred[f"{column_name}_gt"] == True),
    ["PMCID", "repo_url", f"{column_name}_pred", f"{column_name}_gt", "repo_content"]
]

print(fp_is_empty.to_string(index=False))

     PMCID                               repo_url contains_readme_pred  contains_readme_gt                                                                                                                                                                                                                                                                        repo_content
PMC8881932 https://doi.org/10.5281/zenodo.4560078                False                True Repository tree \n {\n  "url": "https://doi.org/10.5281/zenodo.4560078",\n  "tree": [\n    {\n      "name": "epiben",\n      "children": [\n        {\n          "name": "hidosfaikid-v1.0.2.zip",\n          "children": null\n        }\n      ]\n    }\n  ]\n}\nFiles content\n 


In [25]:
column_name = "contains_requirements"
fp_is_empty = df_pred.loc[
    (df_pred[f"{column_name}_pred"] == False) &
    (df_pred[f"{column_name}_gt"] == True),
    ["PMCID", "repo_url", f"{column_name}_pred", f"{column_name}_gt", "repo_content"]
]

print(fp_is_empty.to_string(index=False))

     PMCID                               repo_url contains_requirements_pred  contains_requirements_gt                                                                                                                                                                                                                                                                        repo_content
PMC8881932 https://doi.org/10.5281/zenodo.4560078                      False                      True Repository tree \n {\n  "url": "https://doi.org/10.5281/zenodo.4560078",\n  "tree": [\n    {\n      "name": "epiben",\n      "children": [\n        {\n          "name": "hidosfaikid-v1.0.2.zip",\n          "children": null\n        }\n      ]\n    }\n  ]\n}\nFiles content\n 


In [26]:
column_name = "contains_requirements"
fp_is_empty = df_pred.loc[
    (df_pred[f"{column_name}_pred"] == True) &
    (df_pred[f"{column_name}_gt"] == False),
    ["PMCID", "repo_url", f"{column_name}_pred", f"{column_name}_gt", "repo_content"]
]

print(fp_is_empty.to_string(index=False))

      PMCID                                                                 repo_url contains_requirements_pred  contains_requirements_gt                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [27]:
column_name = "requirements_dependency_versions"
fp_is_empty = df_pred.loc[
    (df_pred[f"{column_name}_pred"] == True) &
    (df_pred[f"{column_name}_gt"] == False),
    ["PMCID", "repo_url", f"{column_name}_pred", f"{column_name}_gt", "repo_content"]
]

print(fp_is_empty.to_string(index=False))

      PMCID                                       repo_url requirements_dependency_versions_pred  requirements_dependency_versions_gt                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [28]:


column_name = "readme_purpose_and_outputs"
fp_is_empty = df_pred.loc[
    (df_pred[f"{column_name}_pred"] == True) &
    (df_pred[f"{column_name}_gt"] == False),
    ["PMCID", "repo_url", f"{column_name}_pred", f"{column_name}_gt", "repo_content"]
]

print(fp_is_empty.to_string(index=False))

      PMCID                                                 repo_url readme_purpose_and_outputs_pred  readme_purpose_and_outputs_gt                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     